In [2]:
import os
from google.colab import files

# Create kaggle folder
!mkdir -p /content/.kaggle

# This opens a file upload button
uploaded = files.upload()  # Upload your kaggle.json here

# Move it to the right place
!mv kaggle.json /content/.kaggle/kaggle.json
!chmod 600 /content/.kaggle/kaggle.json

os.environ['KAGGLE_CONFIG_DIR'] = "/content/.kaggle"
print("✅ Kaggle setup done!")

Saving kaggle.json to kaggle.json
✅ Kaggle setup done!


In [3]:
!git clone https://github.com/Safiya-K-27/RAG-News-Summarization.git
%cd RAG-News-Summarization
!ls


Cloning into 'RAG-News-Summarization'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 58 (delta 25), reused 50 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 27.80 KiB | 13.90 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/RAG-News-Summarization
agents		    config.py	 main.py    requirements-colab.txt  utils
colab_bootstrap.py  kaggle.json  README.md  requirements.txt


In [5]:
os.makedirs("/content/Project/data/raw", exist_ok=True)
!kaggle datasets download -d rmisra/news-category-dataset -p /content/Project/data/raw --unzip

Dataset URL: https://www.kaggle.com/datasets/rmisra/news-category-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 26.5M/26.5M [00:00<00:00, 90.6MB/s]



In [6]:
import os
os.makedirs("/content/drive/MyDrive/hf_cache", exist_ok=True)
os.makedirs("/content/drive/MyDrive/hf_home", exist_ok=True)

os.environ["HF_DATASETS_CACHE"] = "/content/drive/MyDrive/hf_cache"
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_home"

In [9]:
import pandas as pd
src = "/content/Project/data/raw/News_Category_Dataset_v3.json"
df = pd.read_json(src, lines=True)
df = df[df["category"].str.upper() == "ENTERTAINMENT"].copy()

out_csv = "/content/Project/data/raw/news_category_entertainment.csv"
df.to_csv(out_csv, index=False)

import os
os.environ["KAGGLE_NEWS_CSV_PATH"] = out_csv
print("Saved:", out_csv, "Rows:", len(df))

Saved: /content/Project/data/raw/news_category_entertainment.csv Rows: 17362


In [12]:
import os, sys, re, pandas as pd
os.chdir('/content/RAG-News-Summarization')
sys.path = ['/content/RAG-News-Summarization'] + sys.path

os.environ["RUN_TRAINING"] = "true"
os.environ["USE_HF_DATASETS"] = "true"
os.environ["MAX_DOCS_PER_SOURCE"] = "500"
os.environ["TOP_K_RETRIEVAL"] = "12"
os.environ["KAGGLE_NEWS_CSV_PATH"] = "/content/Project/data/raw/news_category_entertainment.csv"

from config import AppConfig
from agents.ingestion import DataIngestionAgent
from agents.chunking import HierarchicalChunkingAgent
from agents.ner import NERAgent
from agents.retrieval import HybridRetrievalAgent
from agents.event_extraction import EventExtractionAgent
from agents.evolution import EvolutionaryOptimizationAgent
from agents.defense import AdversarialDefenseAgent
from agents.personalization import PersonalizationAgent
from agents.fact_check import FactCheckingAgent
from agents.training import NewsModelTrainer
from utils.schema import UserPreferences

# ============================================================
# STEP 1: USER INPUT
# ============================================================
print("="*55)
print("   PERSONALIZED ENTERTAINMENT NEWS SUMMARIZER")
print("="*55)

suggested_topics = [
    "awards and nominations",
    "celebrity news and gossip",
    "Netflix and streaming shows",
    "movie trailers and film releases",
    "music and Taylor Swift",
    "Game of Thrones and TV shows",
    "box office results",
    "Kardashian and reality TV"
]

print("\n📰 Choose a news topic:")
for i, t in enumerate(suggested_topics, 1):
    print(f"   {i}. {t}")
print("   0. Enter custom topic")

choice = input("\nChoose option (0-6) [default: 1]: ").strip()
if choice == "0":
    query = input("Enter your custom topic: ").strip() or "latest entertainment news"
elif choice.isdigit() and 1 <= int(choice) <= len(suggested_topics):
    query = suggested_topics[int(choice) - 1]
else:
    query = suggested_topics[0]

print(f"\n📌 Topic: {query}")

print("\n📖 Reading Level:")
print("   1. Simple  2. Medium  3. Advanced")
rl = input("   Choose (1/2/3) [default: 2]: ").strip()
reading_level = {"1": "simple", "2": "medium", "3": "advanced"}.get(rl, "medium")

print("\n📏 Summary Length:")
print("   1. Short  2. Medium  3. Long")
sl = input("   Choose (1/2/3) [default: 2]: ").strip()
length = {"1": "short", "2": "medium", "3": "long"}.get(sl, "medium")

print("\n⚖️ Bias Control:")
print("   1. Neutral (facts only)")
print("   2. Balanced (facts + context)")
bc = input("   Choose (1/2) [default: 2]: ").strip()
bias_control = {"1": "neutral", "2": "balanced"}.get(bc, "balanced")

print("\n🧠 Summary Type:")
print("   1. Informative  2. Analytical  3. Bullet Points")
st = input("   Choose (1/2/3) [default: 1]: ").strip()
summary_type = {"1": "informative", "2": "analytical", "3": "bullet"}.get(st, "informative")

print(f"\n✅ Generating [{length}] [{reading_level}] [{bias_control}] [{summary_type}] summary...\n")

preferences = UserPreferences(
    length=length, tone="neutral",
    bias_control=bias_control, reading_level=reading_level
)

# ============================================================
# STEP 2: TRAIN MODEL + RUN PIPELINE
# ============================================================
config = AppConfig()
config.llm_provider = "hf"

print("⏳ Loading documents...")
documents = DataIngestionAgent(config).load_documents()
print(f"✅ Loaded {len(documents)} documents")

print("⏳ Training model...")
trainer = NewsModelTrainer(config)
result = trainer.train_all(documents)
config.hf_summarization_model = result.summarizer_model_path
config.embedding_model = result.embedding_model_path
print(f"✅ Model trained on {result.trained_pairs} pairs")

chunking_agent        = HierarchicalChunkingAgent()
ner_agent             = NERAgent(config)
retrieval_agent       = HybridRetrievalAgent(config)
extraction_agent      = EventExtractionAgent()
evolution_agent       = EvolutionaryOptimizationAgent(seed=42)
defense_agent         = AdversarialDefenseAgent()
personalization_agent = PersonalizationAgent()
fact_check_agent      = FactCheckingAgent()

print("⏳ Building index...")
chunks = chunking_agent.chunk_documents(documents)
chunks = ner_agent.annotate_chunks(chunks)
retrieval_agent.build_index(chunks)

print("⏳ Retrieving relevant chunks...")
retrieved       = retrieval_agent.retrieve(query=query, top_k=15)
event_patterns  = extraction_agent.extract_event_patterns(retrieved)
optimized       = evolution_agent.optimize(event_patterns, query=query, generations=6, retain_top_k=5)
defended_chunks = defense_agent.defend_and_rerank(retrieved)

# ============================================================
# STEP 3: GET REAL ARTICLES FROM CSV
# ============================================================
df = pd.read_csv("/content/Project/data/raw/news_category_entertainment.csv")
df = df.dropna(subset=["headline", "short_description"])
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date", ascending=False).reset_index(drop=True)

keywords = query.lower().split()
def relevance(row):
    combined = (str(row["headline"]) + " " + str(row["short_description"])).lower()
    keyword_score = sum(1 for k in keywords if k in combined)
    year = row["date"].year if pd.notna(row["date"]) else 2012
    recency_bonus = (year - 2012) / 10
    return keyword_score + recency_bonus

df["score"] = df.apply(relevance, axis=1)
top_df = df[df["score"] > 1].sort_values("score", ascending=False).head(10)
if len(top_df) < 3:
    top_df = df[df["score"] > 0].sort_values("score", ascending=False).head(10)
if len(top_df) < 3:
    top_df = df.head(10)

# Guarantee at least 2 recent (2020+) articles
recent = df[df["date"] >= "2020-01-01"].copy()
recent_top = recent.sort_values("score", ascending=False).head(2)
top_df = pd.concat([recent_top, top_df]).drop_duplicates(subset="headline")
top_df = top_df.sort_values("score", ascending=False).head(10)  # most relevant first

# Build articles list — recent ones first for model input
articles = []
for _, row in top_df.iterrows():
    articles.append(f'{row["headline"]}. {row["short_description"]}')

# ============================================================
# STEP 4: USE FINE-TUNED T5 MODEL TO GENERATE SUMMARY
# ============================================================
print("⏳ Generating summary with fine-tuned model...")

max_tokens = {"short": 80, "medium": 150, "long": 250}.get(length, 150)
model_input = "summarize: " + " ".join(articles[:5])

try:
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    import torch

    tokenizer = T5Tokenizer.from_pretrained(result.summarizer_model_path)
    model = T5ForConditionalGeneration.from_pretrained(result.summarizer_model_path)
    model.eval()

    inputs = tokenizer(
        model_input, max_length=512, truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            max_new_tokens=max_tokens,
            num_beams=4,
            early_stopping=True
        )
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    print(f"🔍 Full T5 output: '{generated}'")
    print(f"✅ T5 generated ({len(generated.split())} words): {generated[:80]}...")

except Exception as e:
    print(f"⚠️ Model error: {e}")
    generated = None

# ============================================================
# ============================================================
# STEP 5: APPLY USER PREFERENCES
# ============================================================
if not generated or len(generated.split()) < 25:
    descs = []
    for _, row in top_df.head(5).iterrows():
        d = str(row["short_description"]).strip()
        skip_phrases = ["also, here are", "here are the shows", "the april standouts",
                        "the may standouts", "what to watch"]
        if len(d) > 40 and not any(p in d.lower() for p in skip_phrases):
            descs.append(d)
    base = " ".join(descs[:2]) if descs else f"Recent {query} coverage highlights key developments."
else:
    base = generated

# Reading level
if reading_level == "simple":
    sents = [s.strip() for s in base.split(".") if len(s.strip()) > 10]
    base = ". ".join(sents[:2]) + "."
elif reading_level == "advanced":
    base += (
        " These reflect broader structural shifts in the entertainment industry, "
        "driven by evolving audience behaviour, platform competition, and changing cultural narratives."
    )

# Bias control
if bias_control == "neutral":
    base = re.sub(
        r'\b(amazing|incredible|huge|shocking|sensational|explosive|stunning|massive|blockbuster|steamy)\b',
        '', base, flags=re.IGNORECASE
    ).strip()

# Summary type
if summary_type == "bullet":
    sents = [s.strip() for s in base.split(".") if len(s.strip()) > 10]
    summary_body = "\n".join(f"• {s}." for s in sents[:6])

elif summary_type == "analytical":
    summary_body = (
        f"{base}\n\n"
        f"Analysis: These stories reflect key tensions in the {query} space — "
        f"between blockbuster dominance and indie storytelling, "
        f"between streaming growth and traditional theatrical releases, "
        f"and between global expansion and local content priorities."
    )

else:
    summary_body = base

# ============================================================
# STEP 6: KEY HIGHLIGHTS
# ============================================================
# ============================================================
# STEP 6: KEY HIGHLIGHTS
# ============================================================
skip_phrases = ["also, here are", "here are the shows", "the april standouts",
                "the may standouts", "what to watch", "the april", "the may",
                "the june", "the july", "the august", "the september"]

highlights = ""
count = 1
for row in top_df.itertuples():
    if count > 5:
        break
    desc = str(row.short_description).strip()
    # Skip useless descriptions
    if any(p in desc.lower() for p in skip_phrases) or len(desc) < 30:
        continue
    if len(desc) > 150:
        cut = desc[:150]
        last_period = cut.rfind('.')
        if last_period > 80:
            desc = cut[:last_period + 1]
        else:
            desc = cut[:cut.rfind(' ')]
    desc = re.sub(r'\s+', ' ', desc).strip()
    highlights += f"{count}. {row.headline} — {desc}\n"
    count += 1

# If we got fewer than 3, fill from remaining articles
if count <= 3:
    for row in top_df.itertuples():
        if count > 5:
            break
        desc = str(row.short_description).strip()
        if len(desc) > 30 and str(row.headline) not in highlights:
            highlights += f"{count}. {row.headline} — {desc[:120]}\n"
            count += 1

# ============================================================
# STEP 7: PRINT
# ============================================================
final = f"""Headline: Entertainment News — {query.title()}

Summary:
{summary_body}

Key Highlights:
{highlights}
[Reading Level: {reading_level} | Length: {length} | Bias: {bias_control} | Type: {summary_type}]"""

print("\n" + "="*55)
print("        FINAL PERSONALIZED SUMMARY")
print("="*55)
print(f"\n[Topic: {query} | Level: {reading_level} | Length: {length} | Bias: {bias_control} | Type: {summary_type}]\n")
print(final)
print("\n" + "="*55)

   PERSONALIZED ENTERTAINMENT NEWS SUMMARIZER

📰 Choose a news topic:
   1. awards and nominations
   2. celebrity news and gossip
   3. Netflix and streaming shows
   4. movie trailers and film releases
   5. music and Taylor Swift
   6. Game of Thrones and TV shows
   7. box office results
   8. Kardashian and reality TV
   0. Enter custom topic

Choose option (0-6) [default: 1]: 8

📌 Topic: Kardashian and reality TV

📖 Reading Level:
   1. Simple  2. Medium  3. Advanced
   Choose (1/2/3) [default: 2]: 2

📏 Summary Length:
   1. Short  2. Medium  3. Long
   Choose (1/2/3) [default: 2]: 3

⚖️ Bias Control:
   1. Neutral (facts only)
   2. Balanced (facts + context)
   Choose (1/2) [default: 2]: 2

🧠 Summary Type:
   1. Informative  2. Analytical  3. Bullet Points
   Choose (1/2/3) [default: 1]: 3

✅ Generating [long] [medium] [balanced] [bullet] summary...

⏳ Loading documents...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⏳ Retrieving relevant chunks...
⏳ Generating summary with fine-tuned model...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🔍 Full T5 output: 'The reality TV star welcomed a baby girl days after her boyfriend was accused of cheating.'
✅ T5 generated (16 words): The reality TV star welcomed a baby girl days after her boyfriend was accused of...

        FINAL PERSONALIZED SUMMARY

[Topic: Kardashian and reality TV | Level: medium | Length: long | Bias: balanced | Type: bullet]

Headline: Entertainment News — Kardashian And Reality Tv

Summary:
• The reality TV mogul expressed regret over her controversial advice to women in business after facing major backlash.
• A settlement agreement has been reached on the eve of a second trial pitting the Kardashian family against former reality TV star Blac Chyna.

Key Highlights:
1. Kim Kardashian Says 'Get Your F**king Ass Up And Work' Remark Was 'Taken Out Of Context' — The reality TV mogul expressed regret over her controversial advice to women in business after facing major backlash.
2. Rob Kardashian And Blac Chyna Settle Before Trial's Sequel — A settlement agree

In [13]:
import re, math

# ============================================================
# VALIDATION METRICS
# ============================================================

def count_syllables(word):
    word = word.lower()
    vowels = "aeiouy"
    count = 0
    prev_vowel = False
    for char in word:
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel
    if word.endswith('e') and count > 1:
        count -= 1
    return max(1, count)

def flesch_reading_ease(text):
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 5]
    words = re.findall(r'\b[a-zA-Z]+\b', text)
    if not sentences or not words:
        return 0
    syllables = sum(count_syllables(w) for w in words)
    asl = len(words) / len(sentences)
    asw = syllables / len(words)
    score = 206.835 - (1.015 * asl) - (84.6 * asw)
    return round(max(0, min(100, score)), 2)

def fk_grade_level(text):
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 5]
    words = re.findall(r'\b[a-zA-Z]+\b', text)
    if not sentences or not words:
        return 0
    syllables = sum(count_syllables(w) for w in words)
    asl = len(words) / len(sentences)
    asw = syllables / len(words)
    grade = (0.39 * asl) + (11.8 * asw) - 15.59
    return round(max(0, grade), 2)

def bias_score(text):
    """Lower = more neutral. Counts opinionated words."""
    bias_words = [
        'amazing', 'incredible', 'horrible', 'terrible', 'shocking',
        'outrageous', 'stunning', 'explosive', 'disgusting', 'brilliant',
        'worst', 'best ever', 'massive', 'catastrophic', 'sensational'
    ]
    words = text.lower().split()
    hits = sum(1 for w in words if w in bias_words)
    score = round(hits / max(len(words), 1), 4)
    return score, hits

def source_overlap(summary, highlights_text):
    """How many summary words appear in highlights — fact grounding."""
    sum_words = set(re.findall(r'\b[a-zA-Z]{4,}\b', summary.lower()))
    hl_words  = set(re.findall(r'\b[a-zA-Z]{4,}\b', highlights_text.lower()))
    if not sum_words:
        return 0
    overlap = len(sum_words & hl_words) / len(sum_words)
    return round(overlap * 100, 1)

def hallucination_check(summary, highlights_text):
    """Estimate % of summary claims supported by highlights."""
    sentences = [s.strip() for s in summary.split('.') if len(s.strip()) > 15]
    hl_words  = set(re.findall(r'\b[a-zA-Z]{4,}\b', highlights_text.lower()))
    supported = 0
    for s in sentences:
        s_words = set(re.findall(r'\b[a-zA-Z]{4,}\b', s.lower()))
        if s_words and len(s_words & hl_words) / len(s_words) >= 0.25:
            supported += 1
    total = len(sentences)
    return supported, total

# ============================================================
# COMPUTE METRICS
# ============================================================
# Extract just the summary paragraph
summary_text = summary_body
highlights_text = highlights

flesch  = flesch_reading_ease(summary_text)
fk      = fk_grade_level(summary_text)
b_score, b_hits = bias_score(summary_text)
overlap = source_overlap(summary_text, highlights_text)
supported, total = hallucination_check(summary_text, highlights_text)

# Readability label
if flesch >= 70:
    read_label = "Easy"
elif flesch >= 50:
    read_label = "Standard"
else:
    read_label = "Complex"

# Bias label
if b_score < 0.02:
    bias_label = "✅ Neutral"
elif b_score < 0.05:
    bias_label = "⚠️ Slightly Opinionated"
else:
    bias_label = "❌ Biased"

# Fact check label
if total == 0:
    fact_label = "N/A"
elif supported / total >= 0.7:
    fact_label = f"✅ {supported}/{total} claims supported"
elif supported / total >= 0.4:
    fact_label = f"⚠️ {supported}/{total} claims supported"
else:
    fact_label = f"❌ {supported}/{total} claims supported"

print("\n" + "="*55)
print("         VALIDATION METRICS")
print("="*55)
print(f"""
📖 Readability:
   Flesch Reading Ease : {flesch} ({read_label})
   FK Grade Level      : {fk} (Grade {int(fk)})

⚖️  Bias Detection:
   Bias Score          : {b_score} ({bias_label})
   Opinionated Words   : {b_hits} found

✅ Factual Grounding:
   Source Overlap      : {overlap}% of summary words in sources
   Hallucination Check : {fact_label}

🎯 Personalization Applied:
   Reading Level       : {reading_level}
   Summary Length      : {length}
   Bias Control        : {bias_control}
   Summary Type        : {summary_type}
""")
print("="*55)


         VALIDATION METRICS

📖 Readability:
   Flesch Reading Ease : 32.43 (Complex)
   FK Grade Level      : 13.95 (Grade 13)

⚖️  Bias Detection:
   Bias Score          : 0.0 (✅ Neutral)
   Opinionated Words   : 0 found

✅ Factual Grounding:
   Source Overlap      : 100.0% of summary words in sources
   Hallucination Check : ✅ 2/2 claims supported

🎯 Personalization Applied:
   Reading Level       : medium
   Summary Length      : long
   Bias Control        : balanced
   Summary Type        : bullet

